# 🌍 Step 1: Data Collection from USGS API (`01_data_collection.ipynb`)

### What This Notebook Does
1. Fetches 5 full calendar years (`2021-01-01` to `2025-12-31`) of global earthquakes (`minmagnitude=3.0`) from the **USGS Earthquake Hazards Program API**.
2. Queries **month-by-month** (`60` months) so no single request hits the USGS 20,000-row API limit.
3. Fetches **both `csv` and `geojson` formats** for every month and merges them on `"id"` using `pd.merge(..., on="id", how="left")` to obtain all 26 required features.
4. Saves the merged raw dataset to `data/earthquakes_intermediate.csv`.

In [1]:
import os
import datetime as dt
from io import StringIO
import requests
import pandas as pd

# Ensure working directory is the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: ./SeismicProject


### 1.1 Define Helper Function to Get Start & End Date of Each Month

In [2]:
def get_month_range(year, month):
    '''Return (start_date, end_date) strings 'YYYY-MM-DD' for a given year and month.'''
    start_date = dt.date(year, month, 1)
    if month == 12:
        end_date = dt.date(year, 12, 31)
    else:
        end_date = dt.date(year, month + 1, 1) - dt.timedelta(days=1)
    return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")

# Test on January 2021
print("Sample Month Range:", get_month_range(2021, 1))

Sample Month Range: ('2021-01-01', '2021-01-31')


### 1.2 Fetch Monthly CSV + GeoJSON from USGS API and Merge on `id`
- **CSV API Call** gives the 22 tabular columns (`time`, `latitude`, `longitude`, `depth`, `mag`, `place`, etc.).
- **GeoJSON API Call** gives the 6 extra fields (`tsunami`, `sig`, `types`, `ids`, `sources`, `alert`).
- **`pd.merge(csv_df, json_df, on="id", how="left")`** joins them accurately by earthquake ID.

In [3]:
URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
MIN_MAGNITUDE = 3.0
OUTPUT_PATH = "data/earthquakes_intermediate.csv"

current_year = dt.date.today().year
end_year = current_year - 1
start_year = end_year - 4

os.makedirs("data", exist_ok=True)

# Only scrape if earthquakes_intermediate.csv does not already exist
if os.path.exists(OUTPUT_PATH):
    print(f"File '{OUTPUT_PATH}' already exists! Skipping 60-month API download.")
    raw_df = pd.read_csv(OUTPUT_PATH)
    print("Loaded existing intermediate dataset shape:", raw_df.shape)
else:
    total_rows = 0
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            start_str, end_str = get_month_range(year, month)
            csv_params = {
                "format": "csv",
                "starttime": start_str,
                "endtime": end_str,
                "minmagnitude": MIN_MAGNITUDE,
            }
            json_params = {
                "format": "geojson",
                "starttime": start_str,
                "endtime": end_str,
                "minmagnitude": MIN_MAGNITUDE,
            }
            csv_resp = requests.get(URL, params=csv_params, timeout=180)
            json_resp = requests.get(URL, params=json_params, timeout=180)

            csv_df = pd.read_csv(StringIO(csv_resp.text))
            features = json_resp.json().get("features", [])

            json_records = []
            for f in features:
                props = f.get("properties") or {}
                json_records.append({
                    "id": f.get("id"),
                    "tsunami": props.get("tsunami"),
                    "sig": props.get("sig"),
                    "types": props.get("types"),
                    "ids": props.get("ids"),
                    "sources": props.get("sources"),
                    "alert": props.get("alert"),
                })
            json_df = pd.DataFrame(json_records)

            # Merge CSV and GeoJSON on 'id' using Left Join
            merged_df = pd.merge(csv_df, json_df, on="id", how="left")

            write_header = not os.path.exists(OUTPUT_PATH)
            merged_df.to_csv(OUTPUT_PATH, mode="a", index=False, header=write_header)
            total_rows += len(merged_df)
            print(f"{start_str} to {end_str}: saved {len(merged_df)} rows (Total: {total_rows})")
    raw_df = pd.read_csv(OUTPUT_PATH)

File 'data/earthquakes_intermediate.csv' already exists! Skipping 60-month API download.
Loaded existing intermediate dataset shape: (104464, 29)


### 1.3 Inspect the Collected Raw Dataset

In [4]:
raw_df.head()

time  latitude  longitude  ...   nst    gap   dmin
0  2021-01-31 23:20:49.923  -31.7493   -68.9337  ...  32.0   42.0  0.294
1  2021-01-31 23:08:17.161  -15.4902  -177.2052  ...  32.0   64.0  1.471
2  2021-01-31 22:54:19.760   19.7529   121.3159  ...  32.0  106.0  3.057
3  2021-01-31 22:06:00.832   28.1524    57.2570  ...  32.0   71.0  3.330
4  2021-01-31 21:51:14.016   71.3212    -3.7578  ...  32.0   65.0  6.023

[5 rows x 8 columns]
